# Phase 2 · Step 4 — Hybrid search (dense + BM25 sparse)

Recreates `papermind_chunks` with two named vectors per point (`dense` and
`bm25`), re-embeds the same 4 chunks both ways, and combines both rankings
with Reciprocal Rank Fusion. See
`learning/04_rag_pipeline/04_hybrid_search.md` for the why behind these
choices.

## Reload + re-chunk

Same loading/chunking code as notebooks 01 and 03.

In [1]:
from generate_sample_pdf import generate
from llama_index.core.node_parser import SentenceSplitter
from llama_index.readers.file import PDFReader

pdf_path = generate()
documents = PDFReader().load_data(file=pdf_path)
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

print(f"Produced {len(nodes)} chunks from {len(documents)} pages")

Produced 4 chunks from 4 pages


## Recreate the collection with two named vectors

The Step 1 collection only has one unnamed dense vector — switching to a
named `dense` + named `bm25` sparse vector changes that fixed shape, so we
recreate rather than alter in place. `Modifier.IDF` applies IDF weighting on
top of raw BM25 term frequencies, as Qdrant recommends for BM25 sparse
vectors.

In [2]:
from pathlib import Path

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, Modifier, SparseVectorParams, VectorParams

COLLECTION_NAME = "papermind_chunks"
STORAGE_PATH = Path("data/qdrant")
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
SPARSE_MODEL = "Qdrant/bm25"

client = QdrantClient(path=str(STORAGE_PATH))

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={"dense": VectorParams(size=384, distance=Distance.COSINE)},
    sparse_vectors_config={"bm25": SparseVectorParams(modifier=Modifier.IDF)},
)

print(f"Collection exists: {client.collection_exists(COLLECTION_NAME)}")

Collection exists: True


## Embed both ways

In [3]:
from fastembed import SparseTextEmbedding, TextEmbedding

texts = [node.get_content() for node in nodes]

dense_embedder = TextEmbedding(model_name=DENSE_MODEL)
sparse_embedder = SparseTextEmbedding(model_name=SPARSE_MODEL)

dense_vectors = list(dense_embedder.embed(texts))
sparse_vectors = list(sparse_embedder.embed(texts))

print(f"Dense: {len(dense_vectors)} vectors, dim={len(dense_vectors[0])}")
print(f"Sparse: {len(sparse_vectors)} vectors, e.g. {len(sparse_vectors[0].indices)} non-zero terms")

Dense: 4 vectors, dim=384
Sparse: 4 vectors, e.g. 19 non-zero terms


## Build points + upsert

Same `uuid5` ID scheme and payload fields as Step 2 — only the `vector`
shape changes, from a single list to a dict of named vectors.

In [4]:
from uuid import NAMESPACE_URL, uuid5

from qdrant_client.models import PointStruct, SparseVector

page_counters: dict[str, int] = {}
points = []

for node, dense_vector, sparse_vector in zip(nodes, dense_vectors, sparse_vectors):
    page_label = node.metadata["page_label"]
    file_name = node.metadata["file_name"]
    chunk_index = page_counters.get(page_label, 0)
    page_counters[page_label] = chunk_index + 1

    point_id = str(uuid5(NAMESPACE_URL, f"{file_name}:{page_label}:{chunk_index}"))
    points.append(
        PointStruct(
            id=point_id,
            vector={
                "dense": dense_vector.tolist(),
                "bm25": SparseVector(
                    indices=sparse_vector.indices.tolist(),
                    values=sparse_vector.values.tolist(),
                ),
            },
            payload={
                "text": node.get_content(),
                "page_label": page_label,
                "file_name": file_name,
                "chunk_index": chunk_index,
            },
        )
    )

client.upsert(collection_name=COLLECTION_NAME, points=points)

info = client.get_collection(COLLECTION_NAME)
print("Points count:", info.points_count)
assert info.points_count == len(nodes)

Points count: 4


## Hybrid search

Embed the question both ways, run each as a `Prefetch`, and fuse the two
rankings with RRF.

In [5]:
from qdrant_client.models import FusionQuery, Fusion, Prefetch

question = "How is citation accuracy measured in this paper?"

query_dense = next(dense_embedder.embed([question])).tolist()
query_sparse = next(sparse_embedder.embed([question]))

response = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        Prefetch(query=query_dense, using="dense", limit=10),
        Prefetch(
            query=SparseVector(
                indices=query_sparse.indices.tolist(),
                values=query_sparse.values.tolist(),
            ),
            using="bm25",
            limit=10,
        ),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=2,
    with_payload=True,
)

for result in response.points:
    print(f"--- page {result.payload['page_label']} (rrf score={result.score:.3f}) ---")
    print(result.payload["text"])
    print()

--- page 4 (rrf score=0.833) ---
3. Results
Across a held-out set of research papers, grounded answers with page citations were rated
as more trustworthy than ungrounded answers by human reviewers, even when the two answers
conveyed the same information.

--- page 1 (rrf score=0.833) ---
Abstract
This paper studies retrieval-augmented generation for question answering over research
papers. We show that combining dense retrieval with a re-ranking stage improves citation
accuracy compared to retrieval alone.



## Sanity check

Same expectation as Step 3's dense-only search: the "3. Results" page
should still come out on top (or at least in the top 2) once both rankings
are fused.

In [6]:
pages_returned = [r.payload["page_label"] for r in response.points]
print("Pages returned:", pages_returned)
assert any("citation" in r.payload["text"].lower() for r in response.points)
print("A relevant (citation-mentioning) chunk is in the hybrid results.")

Pages returned: ['4', '1']
A relevant (citation-mentioning) chunk is in the hybrid results.


## Next

Phase 2 · Step 5 adds a re-ranker on top of these hybrid results — a slower
but more accurate model that re-scores just the top candidates before
they're handed to the LLM.